# 02 City Reference Model

## Purpose

Create the stable city reference dataset used as the central join anchor across EEA, Wikipedia, Open-Meteo, Kafka, Spark, Gold tables and final analysis.


## Inputs

Local constants for eight selected European cities. No external source is called in this notebook.


## Outputs

- `data/silver/city_reference.csv`
- `data/silver/city_reference.parquet`


## Technologies used

Python, pandas, pyarrow, Jupyter Notebook.


## Configuration

Output paths are project-relative and use `DATA_DIR=data` by default.


In [1]:
from pathlib import Path
import os
import json
import pandas as pd

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))


## Implementation

The city list is intentionally small and reviewable. `city_id` is the only downstream city join key; free-text city names are display fields only.


In [2]:
from pathlib import Path
import pandas as pd

SILVER_DIR = DATA_DIR / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

CITY_RECORDS = [
    {"city_id": "vienna_at", "city_name": "Vienna", "city_name_normalized": "vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Phase 1 pilot; all sources feasible with constraints.", "eea_station_selection_notes": "Selected Vienna station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de", "city_name": "Berlin", "city_name_normalized": "berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Phase 1 pilot; all sources feasible with constraints.", "eea_station_selection_notes": "Selected Berlin station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
    {"city_id": "paris_fr", "city_name": "Paris", "city_name_normalized": "paris", "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Major European capital for geographic spread.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
    {"city_id": "madrid_es", "city_name": "Madrid", "city_name_normalized": "madrid", "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Southern European comparison city.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
    {"city_id": "rome_it", "city_name": "Rome", "city_name_normalized": "rome", "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Mediterranean comparison city.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
    {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "city_name_normalized": "amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Compact urban context and expected monitoring coverage.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
    {"city_id": "warsaw_pl", "city_name": "Warsaw", "city_name_normalized": "warsaw", "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Central/Eastern European comparison city.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
    {"city_id": "prague_cz", "city_name": "Prague", "city_name_normalized": "prague", "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "population": None, "area_km2": None, "population_density": None, "mapping_notes": "Central European comparison city.", "eea_station_selection_notes": "Selected station must be verified against local EEA files.", "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
]

REQUIRED_COLUMNS = ["city_id", "city_name", "city_name_normalized", "country_code", "latitude", "longitude"]

def build_city_reference() -> pd.DataFrame:
    df = pd.DataFrame(CITY_RECORDS)
    expected_city_ids = df["city_name_normalized"] + "_" + df["country_code"].str.lower()
    if not (df["city_id"] == expected_city_ids).all():
        raise ValueError("city_id must follow <city_name_normalized>_<country_code>")
    if not df["city_id"].is_unique:
        raise ValueError("city_id values must be unique")
    if df[REQUIRED_COLUMNS].isna().any().any():
        raise ValueError("required city reference fields must not contain nulls")
    if not df["country_code"].str.fullmatch(r"[A-Z]{2}").all():
        raise ValueError("country_code must use two uppercase letters")
    if not df["latitude"].between(-90, 90).all():
        raise ValueError("invalid latitude")
    if not df["longitude"].between(-180, 180).all():
        raise ValueError("invalid longitude")
    return df

city_reference_df = build_city_reference()
city_reference_df


,city_id,city_name,city_name_normalized,country_code,latitude,longitude,population,area_km2,population_density,mapping_notes,eea_station_selection_notes,wikipedia_url
0,vienna_at,Vienna,vienna,AT,48.2082,16.3738,None,None,None,Phase 1 pilot; all sources feasible with const...,Selected Vienna station must be verified again...,https://en.wikipedia.org/wiki/Vienna
1,berlin_de,Berlin,berlin,DE,52.5200,13.4050,None,None,None,Phase 1 pilot; all sources feasible with const...,Selected Berlin station must be verified again...,https://en.wikipedia.org/wiki/Berlin
2,paris_fr,Paris,paris,FR,48.8566,2.3522,None,None,None,Major European capital for geographic spread.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Paris
3,madrid_es,Madrid,madrid,ES,40.4168,-3.7038,None,None,None,Southern European comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Madrid
4,rome_it,Rome,rome,IT,41.9028,12.4964,None,None,None,Mediterranean comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Rome
5,amsterdam_nl,Amsterdam,amsterdam,NL,52.3676,4.9041,None,None,None,Compact urban context and expected monitoring ...,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Amsterdam
6,warsaw_pl,Warsaw,warsaw,PL,52.2297,21.0122,None,None,None,Central/Eastern European comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Warsaw
7,prague_cz,Prague,prague,CZ,50.0755,14.4378,None,None,None,Central European comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Prague


## Validation / Quality Checks

Validate uniqueness, required fields, country-code format, coordinate ranges and output read-back.


In [3]:
assert len(city_reference_df) == 8
assert city_reference_df["city_id"].is_unique
assert city_reference_df[REQUIRED_COLUMNS].notna().all().all()
assert city_reference_df["latitude"].between(-90, 90).all()
assert city_reference_df["longitude"].between(-180, 180).all()

csv_path = SILVER_DIR / "city_reference.csv"
parquet_path = SILVER_DIR / "city_reference.parquet"
city_reference_df.to_csv(csv_path, index=False)
city_reference_df.to_parquet(parquet_path, index=False)

roundtrip = pd.read_parquet(parquet_path)
assert len(roundtrip) == len(city_reference_df)
assert set(REQUIRED_COLUMNS).issubset(roundtrip.columns)
roundtrip.head()


,city_id,city_name,city_name_normalized,country_code,latitude,longitude,population,area_km2,population_density,mapping_notes,eea_station_selection_notes,wikipedia_url
0,vienna_at,Vienna,vienna,AT,48.2082,16.3738,None,None,None,Phase 1 pilot; all sources feasible with const...,Selected Vienna station must be verified again...,https://en.wikipedia.org/wiki/Vienna
1,berlin_de,Berlin,berlin,DE,52.5200,13.4050,None,None,None,Phase 1 pilot; all sources feasible with const...,Selected Berlin station must be verified again...,https://en.wikipedia.org/wiki/Berlin
2,paris_fr,Paris,paris,FR,48.8566,2.3522,None,None,None,Major European capital for geographic spread.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Paris
3,madrid_es,Madrid,madrid,ES,40.4168,-3.7038,None,None,None,Southern European comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Madrid
4,rome_it,Rome,rome,IT,41.9028,12.4964,None,None,None,Mediterranean comparison city.,Selected station must be verified against loca...,https://en.wikipedia.org/wiki/Rome


## Results

Phase 2 produces a stable city reference CSV and Parquet file for later notebooks.


## Limitations

Coordinates are city-center approximations. Population, area and density are contextual fields and are populated from Wikipedia in Phase 4 where parseable.


## Next step

Run notebook `03_eea_batch_ingestion.ipynb` to process the file/batch EEA source.
